In [ ]:
import base64
import requests
import os
import glob
import json
import time
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
import os.path
from selenium import webdriver
from bs4 import BeautifulSoup
from google.oauth2 import service_account
from typing import List, Optional
import sys
# Define the path to the utils folder
utils_folder = '../utils/'

# Add the utils folder to the system path
sys.path.append(os.path.abspath(utils_folder))
import openai 

In [ ]:
# Load environment variables
load_dotenv()

In [ ]:
# Initialize OpenAI client with API key
client = OpenAI()
OpenAI.api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
class MonetizationData(BaseModel):
    MonetizationMethod: list[str]
    DonationLinks: list[str]
    DonationMethods: list[str]
    SuggestedDonationAmounts: list[str]
    RecurringDonationsOption: str
    TextRelatedToDonations: str
    ButtonsOrWidgets: str
    PlacementOfDonationSection:str
    ThirdPartyPlatforms: list[str]
    CallToActionMessages: list[str]
    PaymentPlatformIntegration:list[str]
    PaymentButtonWidgetLocation:str
    AcceptedPaymentMethods: list[str]
    PricingModels: list[str]
    CurrencySupport: list[str]
    SubscriptionPlans: list[str]   

class MonetizationDataList(BaseModel):
    MonetizationDataList: list['MonetizationData']


class ProductData(BaseModel):
    ProductName: str
    ProductDescription: Optional[str]
    Price: Optional[str]
    DiscountOrOffers: Optional[str]
    ProductCategory: str
    Subcategory: Optional[str]
    ProductPageURL: str
    RelatedProducts: List[str]
    BrandName: str
    SellerInformation: Optional[str]
    ShippingInformation: Optional[str]
    ProductImageURLs: List[str]
    VideoURLs: Optional[List[str]]
    PromotionalBannersOrAds: Optional[str]
    SpecialCampaignsOrDiscounts: Optional[str]
    ProductTagsOrKeywords: Optional[List[str]]
    WebpageTitle: str
    MetaDescription: Optional[str]
    LastUpdatedDate: str
    PublisherOrWebsiteOwner: str

class ProductDataList(BaseModel):
    ProductDataList: List[ProductData]

In [ ]:
def generate_prompt(page_text, links, prompt_type):
    """
    Generates a structured prompt based on the type of information to extract 
    from a webpage, such as donations, product sales, or advertising details.

    Args:
        page_text (str): The text content of the webpage.
        links (str): Links found on the webpage.
        prompt_type (str): Type of prompt to generate ("donations", "product_sales", or "advertisings").

    Returns:
        str: A formatted prompt string for extracting specific data from the webpage.
    """
    # Generate prompt for donation-related information extraction
    if prompt_type == "donations":
        prompt = f"""
        You are an expert in identifying payment and donation-related data from websites. 
        The website may be in English or another language. Your task is to extract the 
        following monetization-related information, translate all text into English, and 
        return it in the format provided below:

        - **MonetizationMethod:** [Type of monetization method used (e.g., donations, subscriptions, pay-per-article, ads, etc.)]
        - **DonationLinks:** [URLs directing users to donation pages or forms]
        - **DonationMethods:** [Types of donation methods accepted (e.g., PayPal, credit cards, cryptocurrency)]
        - **SuggestedDonationAmounts:** [Any preset or suggested donation amounts]
        - **RecurringDonationsOption:** [Information on one-time vs recurring donation options]
        - **TextRelatedToDonations:** [Specific phrases or text promoting donations (e.g., 'Support us', 'Donate now'), translated into English]
        - **ButtonsOrWidgets:** [Visual elements facilitating donations (e.g., donation buttons, widgets)]
        - **PlacementOfDonationSection:** [Location of donation-related elements on the page (e.g., header, footer, sidebar)]
        - **ThirdPartyPlatforms:** [Integrations with third-party platforms (e.g., Patreon, Ko-fi, GoFundMe)]
        - **CallToActionMessages:** [Prominent messages encouraging donations, translated into English]
        - **PaymentPlatformIntegration:** [Payment platforms integrated on the website (e.g., Stripe, PayPal, Square)]
        - **PaymentButtonWidgetLocation:** [Location of payment buttons or widgets (e.g., top, bottom, sidebar)]
        - **AcceptedPaymentMethods:** [Methods accepted for payments (e.g., credit cards, bank transfers, cryptocurrency)]
        - **PricingModels:** [Pricing models used (e.g., pay-per-article, subscriptions, freemium models)]
        - **CurrencySupport:** [Currencies supported for payments or donations]
        - **SubscriptionPlans:** [Details about subscription plans (e.g., pricing tiers, duration, benefits), translated into English]

        Below is the text scraped from the webpage:

        **Webpage text:**
        {page_text}

        Additionally, here are the links found on the page. Please extract and validate any links related to payments or donations:
        {links}
        """
    # Generate prompt for product sales-related information extraction
    elif prompt_type == "product_sales":
        prompt = f"""
        You are given the text of a webpage that lists products sold online. 
        Your task is to extract specific data from the webpage, structured as follows:
           - Product Name
           - Product Description (if available)
           - Price (if available)
           - Discount or Offers (if available)
           - Product Category
           - Subcategory (if available)
           - Product Page URL
           - Related Products or Links to Other Products
           - Brand Name
           - Seller Information (if applicable)
           - Shipping Information (if mentioned)
           - Product Image URL(s)
           - Video URL(s) (if present)
           - Promotional Banners or Ads (if present)
           - Special Campaigns or Discounts
           - Product Tags or Keywords (if listed)
           - Webpage Title
           - Meta Description (if available)
           - Last Updated Date
           - Publisher or Website Owner

        The webpage text is:
        {page_text}

        Additionally, here are the links found on the page:
        {links}

        Please extract the information and present it in a structured format.
        """
        
    
    return prompt


In [ ]:
def generate_completions(prompt, prompt_type, model='gpt-4o-mini', max_tokens=20000):
    """
    Sends a request to the OpenAI API to generate completions based on the given prompt.
    
    Args:
        prompt (str): A text prompt to be used by the model.
        prompt_type (str): The type of prompt (e.g., "donations", "product_sales", "advertisings").
        model (str): The model name to be used. Defaults to 'gpt-4o-mini'.
        max_tokens (int): The maximum number of tokens to generate. Defaults to 20000.

    Returns:
        dict: A parsed response containing the generated data.
    """ 
    # Define the response structure based on prompt type
    if prompt_type == "donations":
        response_structure = MonetizationDataList
    elif prompt_type == "product_sales":
        response_structure = ProductDataList
    elif prompt_type == "advertisings":
        response_structure = AdvertisingDataList
    else:
        raise ValueError("Invalid prompt_type. Must be one of: 'donations', 'product_sales', 'advertisings'.")

    # Send request to the OpenAI API with structured output parsing
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [{"type": "text", "text": prompt}]
            }
        ],
        response_format=response_structure,
        max_tokens=max_tokens,
        temperature=0
    )
    
    return completion.choices[0].message.parsed

In [ ]:
def create_dataframe(df, prompt_type, monetization_url, source_url):
    """
    Extract monetization data from websites and store it in a DataFrame.

    Args:
        df (pd.DataFrame): DataFrame containing 'Monetization_url' and 'Original_url' columns.
        prompt_type (str): Type of prompt to be generated, indicating the kind of information to be extracted
                           (e.g., 'donations', 'product_sales', 'advertisings').
        monetization_url (str): Column name for URLs pointing to monetization-related pages.
        source_url (str): Column name for source URLs from which data is being extracted.

    Returns:
        pd.DataFrame: DataFrame with extracted monetization information.
    """
    data = []

    for index, row in df.iterrows():
        try:
            # Initialize WebDriver to fetch webpage content with JavaScript rendering
            driver = webdriver.Chrome()
            driver.get(row[monetization_url])  # Load the page

            # Get the page source after JavaScript rendering
            html = driver.page_source
            driver.quit()

            # Parse HTML content
            soup = BeautifulSoup(html, "html.parser")

            # Extract all text and links from the webpage
            page_text = soup.get_text()  # Extracts all text from the webpage
            links = [a['href'] for a in soup.find_all('a', href=True)]  # Extracts all hyperlinks

            # Generate prompt and get completions from the model
            prompt = generate_prompt(page_text, links, prompt_type)
            results = generate_completions(prompt, prompt_type, model='gpt-4o-mini', max_tokens=15000)

            if prompt_type == "donations":
                # Process each result from the model output for donations
                for monetization_data_item in results.MonetizationDataList:
                    donation_dict = {
                        'Source_Url': row[source_url],
                        'Monetization_Url': row[monetization_url],
                        'Monetization_Method': monetization_data_item.MonetizationMethod,
                        'Donation_Links': monetization_data_item.DonationLinks,
                        'Donation_Methods': monetization_data_item.DonationMethods,
                        'Suggested_Donation_Amounts': monetization_data_item.SuggestedDonationAmounts,
                        'Recurring_Donations_Option': monetization_data_item.RecurringDonationsOption,
                        'Text_Related_To_Donations': monetization_data_item.TextRelatedToDonations,
                        'Buttons_Or_Widgets': monetization_data_item.ButtonsOrWidgets,
                        'Placement_Of_Donation_Section': monetization_data_item.PlacementOfDonationSection,
                        'ThirdParty_Platforms': monetization_data_item.ThirdPartyPlatforms,
                        'Call_To_Action_Messages': monetization_data_item.CallToActionMessages,
                        'PaymentPlatform_Integration': monetization_data_item.PaymentPlatformIntegration,
                        'Payment_Button_Widget_Location': monetization_data_item.PaymentButtonWidgetLocation,
                        'Accepted_Payment_Methods': monetization_data_item.PricingModels,
                        'Pricing_Models': monetization_data_item.CurrencySupport,
                        'Currency_Support': monetization_data_item.SubscriptionPlans,
                        'Subscription_Plans': monetization_data_item.SubscriptionPlans
                    }
                    data.append(donation_dict)
            
            elif prompt_type == "product_sales":
                # Process each result from the model output for product sales
                for product_data_item in results.ProductDataList:
                    product_dict = {
                        'Source_Url': row[source_url],
                        'Monetization_Url': row[monetization_url],
                        'Product_Name': product_data_item.ProductName,
                        'Product_Description': product_data_item.ProductDescription,
                        'Price': product_data_item.Price,
                        'Discount_Or_Offers': product_data_item.DiscountOrOffers,
                        'Product_Category': product_data_item.ProductCategory,
                        'Subcategory': product_data_item.Subcategory,
                        'Product_Page_URL': product_data_item.ProductPageURL,
                        'Related_Products': product_data_item.RelatedProducts,
                        'Brand_Name': product_data_item.BrandName,
                        'Seller_Information': product_data_item.SellerInformation,
                        'Shipping_Information': product_data_item.ShippingInformation,
                        'Product_Image_URLs': product_data_item.ProductImageURLs,
                        'Video_URLs': product_data_item.VideoURLs,
                        'Promotional_Banners_Or_Ads': product_data_item.PromotionalBannersOrAds,
                        'Special_Campaigns_Or_Discounts': product_data_item.SpecialCampaignsOrDiscounts,
                        'Product_Tags_Or_Keywords': product_data_item.ProductTagsOrKeywords,
                        'Webpage_Title': product_data_item.WebpageTitle,
                        'Meta_Description': product_data_item.MetaDescription,
                        'Last_Updated_Date': product_data_item.LastUpdatedDate,
                        'Publisher_Or_Website_Owner': product_data_item.PublisherOrWebsiteOwner
                    }
                    data.append(product_dict)

        except requests.exceptions.RequestException as e:
            print(f"Error fetching {row[monetization_url]}: {e}")
            continue
        except Exception as e:
            print(f"Error processing data for {row[monetization_url]}: {e}")
            continue

    # Return the DataFrame with collected monetization data
    return pd.DataFrame(data)

In [ ]:
# Specify the file path
filepath = "../data/monetisation_urls_datasets.xlsx"

# Load the Excel file into a DataFrame
df = pd.read_excel(filepath)

# Display the first few rows to verify
print(df.head())


In [ ]:
# Drop rows with missing values in the 'Monetization_Url' column
df = df.dropna(subset=['Monetization_Url'])

# Define the revenue types to exclude
exclude_types = ['advertising', 'partner funded', 'product sale', 'magazine sale']

# Filter out rows where 'Type_Of_Revenue' matches any of the excluded types (case insensitive)
df1 = df[~df['Type_Of_Revenue'].astype(str).str.lower().isin(exclude_types)]

# Verify the filtered DataFrame
df1.shape

In [ ]:
import ast

# Generate the initial DataFrame with donation-related data
final_df = create_dataframe(df1, "donations", 'Monetization_Url', 'Original_Url')

# Convert string representations of lists in 'Donation_Links' to actual list objects
final_df['Donation_Links'] = final_df['Donation_Links'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Filter rows where 'Donation_Links' contains at least one link, then explode the list into separate rows
df_exploded = final_df[final_df['Donation_Links'].apply(lambda x: isinstance(x, list) and len(x) > 0)].explode('Donation_Links')

# Run create_dataframe again on the exploded DataFrame, using 'Donation_Links' as the monetization URL
results_df = create_dataframe(df_exploded, "donations", 'Donation_Links', 'Source_Url')

# Display the results DataFrame
print(results_df.head())

In [ ]:
# Concatenate the two DataFrames
df_concat = pd.concat([final_df, results_df], ignore_index=True)

# Save the combined DataFrame to a CSV file
output_path = "../data/monetization_dataset_final.csv"
df_concat.to_csv(output_path, index=False)

# Print confirmation message
print(f"Data successfully saved to {output_path}")

In [ ]:
# List of revenue types to filter from the main DataFrame
key_list = ['Magzine Sale', 'Advertising', 'Patner Funded', 'Product Sale', 'Patner funded']

# Filter df to include only rows with 'Type_Of_Revenue' values in key_list
df2 = df[df['Type_Of_Revenue'].isin(key_list)]

# Separate DataFrame for 'Product Sale' and 'Magzine Sale' types
product_sales_df = df2[df2['Type_Of_Revenue'].isin(['Product Sale', 'Magzine Sale'])]

# Separate DataFrame for 'Advertising' type
advertising_df = df2[df2['Type_Of_Revenue'] == 'Advertising']

# Print confirmation to check the number of rows in each filtered DataFrame
print(f"Product Sales DataFrame rows: {len(product_sales_df)}")
print(f"Advertising DataFrame rows: {len(advertising_df)}")

In [ ]:
# Extract data from product_sales_df using the 'product_sales' prompt type
product_sales_extracted_data_df = create_dataframe(
    product_sales_df, "product_sales", 'Monetization_Url', 'Original_Url'
)

# Save the extracted data to a CSV file
product_sales_extracted_data_df.to_csv("../data/product_sales_extracted_data_final.csv", index=False)

print("Product sales data extraction completed and saved to product_sales_extracted_data.csv")

In [ ]:
product_sales_extracted_data_df.to_csv("../data/product_sales_extracted_data.csv", index=False)
advertising_df.to_csv("../data/advertising_extracted_data.csv", index=False)

In [ ]:
type(results_df)

In [ ]:
# Mark all duplicates as True
duplicates = df_concat['Monetization_Url'].duplicated(keep=False)
df[duplicates]

In [ ]:
mon_df = pd.read_csv("../data/monetization_dataset_final.csv")

In [ ]:
mon_df.shape

In [ ]:
def clean_string_of_list(column):
    def parse_and_convert(value):
        try:
            # Safely evaluate the string as a Python literal
            parsed = ast.literal_eval(value)
            # Check if it's a list, then join as comma-separated values
            if isinstance(parsed, list):
                return ", ".join(map(str, parsed))
        except (ValueError, SyntaxError):
            # Return original value if parsing fails
            return value
    return column.apply(parse_and_convert)

In [ ]:
import ast

In [ ]:
mon_df['Monetization_Method'] = clean_string_of_list(mon_df['Monetization_Method'])
mon_df['Donation_Links'] = clean_string_of_list(mon_df['Donation_Links'])
mon_df['Donation_Methods'] = clean_string_of_list(mon_df['Donation_Methods'])
mon_df['ThirdParty_Platforms'] = clean_string_of_list(mon_df['ThirdParty_Platforms'])
mon_df['Call_To_Action_Messages'] = clean_string_of_list(mon_df['Call_To_Action_Messages'])
mon_df['PaymentPlatform_Integration'] = clean_string_of_list(mon_df['PaymentPlatform_Integration'])
mon_df['Accepted_Payment_Methods'] = clean_string_of_list(mon_df['Accepted_Payment_Methods'])
mon_df['Pricing_Models'] = clean_string_of_list(mon_df['Pricing_Models'])
mon_df['Currency_Support'] = clean_string_of_list(mon_df['Currency_Support'])
mon_df['Subscription_Plans'] = clean_string_of_list(mon_df['Subscription_Plans'])
mon_df['Suggested_Donation_Amounts'] = clean_string_of_list(mon_df['Suggested_Donation_Amounts'])

In [ ]:
mon_df.duplicated(keep=False).sum()

In [ ]:
dd =  mon_df.drop_duplicates()

In [ ]:
dd.to_csv("ty_without_duplicates.csv",index=False,encoding = 'utf-8')

In [ ]:
len(dd)

In [ ]:
dd1 = dd.drop_duplicates(subset= ['Source_Url','Monetization_Url','Placement_Of_Donation_Section'],keep='first')

In [ ]:
dd2 = dd1.drop(columns = ['Accepted_Payment_Methods','ThirdParty_Platforms','PaymentPlatform_Integration'])

In [ ]:
dd2 =pd.read_csv("ty2_no_dup.csv",encoding = 'utf-8')

In [ ]:
dd1

In [ ]:
dd2 = dd1.drop(columns = ['Donation_Links'])